<a href="https://colab.research.google.com/github/WhoisMonesh/Colab-Debrid-Downloader/blob/main/Colab-Debrid-Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-Debrid Torrent Downloader

Offload torrents to Real-Debrid and download directly to Google Drive.


### 1. Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install Dependencies

In [0]:
!pip install requests -q
print('Dependencies installed.')

### 3. Configuration

In [0]:
# === CONFIGURATION ===
SAVE_PATH = '/content/downloads/Real-DebridTorrentDownloader/'  # local temp dir
DRIVE_PATH = '/content/drive/My Drive/Real-DebridTorrentDownloader/'  # final destination

API_TOKEN = ""  # Real-Debrid API token
MAGNET_LINK = ""  # Magnet link to download

KEEP_ALIVE = True  # prevent Colab timeout

# === END CONFIGURATION ===

### 4. Download

In [0]:
import os, time, datetime, shutil, glob
from google.colab import files

def format_bytes(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'
def main():
    os.makedirs(SAVE_PATH, exist_ok=True)
    print(f'Save path: {SAVE_PATH} (local)')
    if KEEP_ALIVE:
        from IPython.display import display, Javascript
        display(Javascript("""function keepAlive(){var btn=document.querySelector(\"colab-connect-button\");if(btn)btn.click()}setInterval(keepAlive,120000);"""))
        print('Keep-alive active')
    begin = time.time()
    from IPython.display import display, HTML
    progress_display = display(HTML('<pre>Starting...</pre>'), display_id='dl-progress')

    import requests, time
    def debrid_download(token, magnet, save_path):
        headers = {"Authorization": f"Bearer {token}"}
        r = requests.post("https://api.real-debrid.com/rest/1.0/torrents/addMagnet", headers=headers, data={"magnet": magnet})
        tid = r.json().get("id")
        print(f"Torrent added: {tid}")
        requests.post(f"https://api.real-debrid.com/rest/1.0/torrents/selectFiles/{tid}", headers=headers, data={"files": "all"})
        print("Waiting for processing...")
        while True:
            r = requests.get(f"https://api.real-debrid.com/rest/1.0/torrents/info/{tid}", headers=headers)
            if r.json().get("status") == "downloaded":
                break
            time.sleep(10)
        links = r.json().get("links", [])
        count = 0
        for link in links:
            r = requests.post("https://api.real-debrid.com/rest/1.0/unrestrict/link", headers=headers, data={"link": link})
            dl_url = r.json().get("download")
            fname = r.json().get("filename", f"file_{count}")
            print(f"Downloading: {fname}")
            r = requests.get(dl_url, stream=True)
            with open(os.path.join(save_path, fname), "wb") as f:
                for chunk in r.iter_content(8192):
                    if chunk: f.write(chunk)
            count += 1
            print(f"  Done")
        return count
    if not API_TOKEN:
        print("ERROR: API_TOKEN required from https://real-debrid.com/apitoken")
        return
    count = debrid_download(API_TOKEN, MAGNET_LINK, SAVE_PATH)
    print(f"Downloaded {count} files")
    end = time.time()
    elapsed = end - begin
    print(f'\n{"="*50}')
    print('COMPLETE')
    print(f'Elapsed: {int(elapsed//60)}m {int(elapsed%60)}s')
    print(f'Saved locally: {SAVE_PATH}')

    items = glob.glob(os.path.join(SAVE_PATH, '*'))
    items = [f for f in items if os.path.isfile(f)]
    if len(items) > 1:
        import zipfile
        total = sum(os.path.getsize(f) for f in items)
        processed = 0
        zpath = SAVE_PATH.rstrip('/').rstrip('\\') + '.zip'
        print(f'\nZipping {len(items)} files...')
        with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fpath in items:
                zf.write(fpath, os.path.basename(fpath))
                processed += os.path.getsize(fpath)
                pct = int(processed * 100 / total) if total else 0
                bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
                progress_display.update(HTML(f'<pre>Zipping: |{bar}| {pct}% | {os.path.basename(fpath)}</pre>'))
        print(f'Zip: {zpath}')
        print(f'Size: {format_bytes(os.path.getsize(zpath))}')
        display(HTML(f'<a href="{zpath}" download>Download ZIP</a>'))
        files.download(zpath)

    print(f'\nMoving to Drive...')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for f in os.listdir(SAVE_PATH):
        shutil.move(os.path.join(SAVE_PATH, f), os.path.join(DRIVE_PATH, f))
    print(f'Final: {DRIVE_PATH}')
    print(f'{"="*50}')
if __name__ == '__main__':
    main()
